In [ ]:
# from datasets import load_dataset

# # Загрузка датасета
# dataset = load_dataset("Vikhrmodels/russian_math")

# # Просмотр структуры
# print(dataset)
# print(dataset['train'][0])  # Пример первой записи

# # Сохранение локально (опционально)
# #dataset.save_to_disk("./russian_math_local")

Generating train split: 100%|██████████| 199/199 [00:00<00:00, 66070.33 examples/s]

DatasetDict({
    train: Dataset({
        features: ['task', 'solution', 'short answer', 'class', 'grade'],
        num_rows: 199
    })
})
{'task': 'Девять действительных a1, a2, ..., a9 образуют арифметическую прогрессию. Известно, что a9 в 3 раза больше среднего арифметического этих девяти чисел. Найдите a1, если известно, что a4 = 6.', 'solution': 'Пусть 𝑎 — первый член прогрессии, а 𝑑 — её разность, тогда девять членов прогрессии равны 𝑎, 𝑎 + 𝑑, 𝑎 + 2𝑑, …, 𝑎 + 8𝑑. \r\nСреднее арифметическое чисел в арифметической прогрессии, состоящей из нечётного числа членов, равно среднему из этих чисел, т. е. в данном случае 𝑎 + 4𝑑. \r\nПолучаем уравнение 𝑎 + 8𝑑 = 3(𝑎 + 4𝑑), откуда следует 𝑎 + 8𝑑 = 3𝑎 + 12𝑑 и 𝑎 = −2𝑑. Тогда 6 = 𝑎4 = 𝑎 + 3𝑑 = 𝑑. Значит, 𝑎1 = −2𝑑 = −12.', 'short answer': '-12', 'class': 'school', 'grade': 11}


# Отладка агентов

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

MODEL_ID = os.getenv("MODEL_ID", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "512"))
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

_llm = None

def get_llm():
    global _llm
    if _llm is None:
        import torch
        from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
        from huggingface_hub import login
        
        # Логинимся если есть токен
        if HUGGINGFACE_TOKEN:
            login(token=HUGGINGFACE_TOKEN)
        
        torch.cuda.empty_cache()
        
        pipeline = HuggingFacePipeline.from_model_id(
            model_id=MODEL_ID,
            task="text-generation",
            device=0,
            pipeline_kwargs=dict(
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.1,
            ),
        )
        _llm = ChatHuggingFace(llm=pipeline)
    
    return _llm

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
#from config import get_llm

# Промпт на русском, без служебных токенов
SYSTEM_PROMPT = """Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]"""

TOPIC_PROMPTS = {
    "алгебра": "Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.",
    "комбинаторика": "Составь задачу по комбинаторике: перестановки, сочетания или размещения.",
    "вероятность и статистика": "Составь задачу по теории вероятностей или статистике."
}

def extract_problem_answer(text: str) -> tuple:
    """Извлекает задачу и ответ из ответа модели"""
    # Убираем служебные токены если они есть
    text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
    
    problem = ""
    answer = ""

    if "ЗАДАЧА:" in text and "ОТВЕТ:" in text:
        parts = text.split("ОТВЕТ:")
        if len(parts) >= 2:
            problem_part = parts[0]
            answer = parts[1].strip().split("\n")[0]
            problem = problem_part.replace("ЗАДАЧА:", "").strip()
    
    if not problem:
        lines = [l.strip() for l in text.split("\n") if l.strip() and not l.startswith("Ты ") and not l.startswith("Требования")]
        if len(lines) >= 2:
            problem = "\n".join(lines[:-1])
            answer = lines[-1]
        else:
            problem = text
            answer = "неизвестно"
    
    return problem, answer

class GeneratorAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def generate(self, topic: str) -> dict:
        topic_hint = TOPIC_PROMPTS.get(topic, "Составь математическую задачу.")
        
        system_msg = SystemMessage(content=SYSTEM_PROMPT)
        human_msg = HumanMessage(content=f"Тема: {topic}. {topic_hint}")
        
        response = self.llm.invoke([system_msg, human_msg])
        text = response.content
        
        return response

In [3]:
agentGenerator = GeneratorAgent()

/home/tas/.cache/pypoetry/virtualenvs/ai-mas-hse-project-VtkJAIkS-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.18it/s]
Device set to use cuda:0


In [4]:
agent_answer = agentGenerator.generate("алгебра")

In [11]:
print(agent_answer.content)

<|startoftext|><|im_start|>system
Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]<|im_end|>
<|im_start|>user
Тема: алгебра. Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.<|im_end|>
<|im_start|>assistant
ЗАДАЧА: Уравнение числовых линейных операций: найди значение \( x \) так что  
\[ 4(2x - 5) + 7 = 3(x + 1) - 2x \]
ОТВЕТ: 3

---

Как разумно подумать? Решить уравнение как только возможное с упрощения обхваток.  
[3]


In [11]:
print(agent_answer['full_response'])

<|startoftext|><|im_start|>system
Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]<|im_end|>
<|im_start|>user
Тема: алгебра. Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.<|im_end|>
<|im_start|>assistant
ZÁDACH: Число *x* такое, что в решении системы уравнений имеет простое решение. Если уравнения будут так:  
\[
\begin{cases}
x + 2y = 7 \\
3x - y = 5
\end{cases}
\], подставляя выражение за *y* из второго уравнения во первого — как показать, количество решений в зависимости от определения *x*. Но для специального случая: найти значение *x*, при котором система имеет **решенное** (простое) решение. Поскольку это систему прямых процентов, её определить как точку пересечения.  

Можно попробовать другое подходе.  

---

In [ ]:
import pickle

with open("../data/final_dataset/list_dict_with_tasks.pkl", "rb") as file:
    list_json = pickle.load(file)

In [15]:
import pandas as pd

df = pd.DataFrame(list_json)

In [31]:
df.head()

,id,topic,subtopic,complexity_level,problem,solution,answer
0,115968,Алгебра и арифметика,Исследование квадратного трехчлена,2+,"Известно, что разность кубов корней квадратног...","Из условия следует, что первое уравнение имеет...",Два корня
1,64824,Алгебра и арифметика,Текстовые задачи (прочее),3,"Три пирата нашли клад, состоящий из 240 золоты...","Пусть один из слитков стоит 121 доллар, а кажд...",Может
2,31235,Алгебра и арифметика,Арифметика остатков (прочее),2,Найти последнюю цифру числа 1·2 + 2·3 + ... + ...,1·2 + 2·3 + ... + 999·1000 ≡ 1·2 + 2·3 + ... +...,0
3,116875,Алгебра и арифметика,Текстовые задачи (прочее),3,"Туристическая фирма провела акцию: ""Купи путев...","Каждый из x ""счастливчиков"" привёл по 4 друга....",29 туристов
4,30862,Алгебра и арифметика,Неравенство Коши,2,"Докажите, что ½ ( x ² + y ²) ≥ xy при любых x ...","Перегруппировав члены, получаем ½ ( x – y )² ≥ 0.",


In [17]:
df.topic.value_counts()

topic
Алгебра и арифметика        71
Комбинаторика               46
Олимпиадные задачи          13
Вероятность и статистика    12
Математический анализ        5
Name: count, dtype: int64

In [33]:
random_task = df.sample(1)

text_task = random_task['problem'].iloc[0]
answer_task = random_task['answer'].iloc[0]

In [34]:
text_task, answer_task

('Изменятся ли частное и остаток, если делимое и делитель увеличить в 3 раза?',
 'Частное не изменится, остаток изменится, если он был отличен от нуля')

---

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage
from config import get_llm

SYSTEM_PROMPT = """Ты проверяющий. Сравни ответ ученика с правильным ответом.

Ответь одним словом:
ПРАВИЛЬНО — если ответы совпадают (допускаются небольшие отличия в формате)
НЕПРАВИЛЬНО — если ответы разные

Не объясняй, просто скажи ПРАВИЛЬНО или НЕПРАВИЛЬНО."""

class ReviewerAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def review(self, solver_answer: str, ground_truth: str = None) -> dict:
        gt_text = ground_truth if ground_truth else "не предоставлен"
        
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Ответ ученика: {solver_answer}\nПравильный ответ: {gt_text}")
        ]
        
        response = self.llm.invoke(messages)
        text = response.content.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip().upper()
        
        is_correct = "ПРАВИЛЬНО" in text or "CORRECT" in text
        
        return {
            "full_response": response.content,
            "verdict": "ПРАВИЛЬНО" if is_correct else "НЕПРАВИЛЬНО",
            "is_correct": is_correct
        }

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage
from config import get_llm

SYSTEM_PROMPT = """Ты репетитор по математике. Реши задачу пошагово.

Формат ответа (строго):
РЕШЕНИЕ: [пошаговое решение]
ОТВЕТ: [итоговый числовой ответ]

Будь кратким. ОТВЕТ — только число или формула."""

def extract_answer(text: str) -> str:
    """Извлекает ответ из решения"""
    # Убираем служебные токены
    text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
    
    if "ОТВЕТ:" in text:
        parts = text.split("ОТВЕТ:")
        if len(parts) >= 2:
            return parts[1].strip().split("\n")[0]
    
    if "ANSWER:" in text:
        parts = text.split("ANSWER:")
        if len(parts) >= 2:
            return parts[1].strip().split("\n")[0]
    
    # Fallback — последняя строка
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    return lines[-1] if lines else text.strip()

class SolverAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def solve(self, problem: str) -> dict:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Реши задачу: {problem}")
        ]
        
        response = self.llm.invoke(messages)
        text = response.content #.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
        extracted = extract_answer(text)
        
        return {
            "full_response": text,
            "answer": extracted
        }

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage
from config import get_llm
from pathlib import Path
import pickle
import os
import pandas as pd

# Промпт на русском, без служебных токенов
SYSTEM_PROMPT = """Ты генератор математических задач. Придумай ОДНУ задачу по указанной теме.

Требования:
1. Задача должна быть реалистичной и понятной
2. Ответ — число или простая формула
3. Сложность: уровень ЕГЭ или школьной олимпиады

Формат ответа (строго соблюдай):
ЗАДАЧА: [текст задачи]
ОТВЕТ: [числовой ответ]"""

TOPIC_PROMPTS = {
    "алгебра": "Составь задачу по алгебре: уравнения, неравенства, функции или прогрессии.",
    "комбинаторика": "Составь задачу по комбинаторике: перестановки, сочетания или размещения.",
    "вероятность и статистика": "Составь задачу по теории вероятностей или статистике."
}

def extract_problem_answer(text: str) -> tuple:
    """Извлекает задачу и ответ из ответа модели"""
    # Убираем служебные токены если они есть
    text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
    
    problem = ""
    answer = ""
    
    # Ищем по-русски
    if "ЗАДАЧА:" in text and "ОТВЕТ:" in text:
        parts = text.split("ОТВЕТ:")
        if len(parts) >= 2:
            problem_part = parts[0]
            answer = parts[1].strip().split("\n")[0]
            problem = problem_part.replace("ЗАДАЧА:", "").strip()
    # Fallback на английский если модель переключилась
    elif "PROBLEM:" in text and "ANSWER:" in text:
        parts = text.split("ANSWER:")
        if len(parts) >= 2:
            problem = parts[0].replace("PROBLEM:", "").strip()
            answer = parts[1].strip().split("\n")[0]
    
    # Если не распарсилось — берём всё как задачу, последнее число как ответ
    if not problem:
        lines = [l.strip() for l in text.split("\n") if l.strip() and not l.startswith("Ты ") and not l.startswith("Требования")]
        if len(lines) >= 2:
            problem = "\n".join(lines[:-1])
            answer = lines[-1]
        else:
            problem = text
            answer = "неизвестно"
    
    return problem, answer

class GeneratorAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def generate(self, topic: str) -> dict:
        topic_hint = TOPIC_PROMPTS.get(topic, "Составь математическую задачу.")
        
        # Формируем сообщения вручную в формате TinyLlama
        # TinyLlama использует: <|system|>...<|user|>...<|assistant|>...
        
        system_msg = SystemMessage(content=SYSTEM_PROMPT)
        human_msg = HumanMessage(content=f"Тема: {topic}. {topic_hint}")
        
        response = self.llm.invoke([system_msg, human_msg])
        text = response.content
        
        # # Очистка от служебных токенов
        # text = text.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip()
        
        problem, answer = extract_problem_answer(text)
        
        return {
            "full_response": text,
            "problem": problem,
            "ground_truth": answer
        }
    
    def generate_static_task(self, topic: str):
        # Путь внутри контейнера
        dataset_path = Path("/app/static_dataset/list_dict_with_tasks.pkl")
        
        with open(dataset_path, "rb") as f:
            df = pickle.load(f)

        df = pd.DataFrame(df)

        random_task = df.sample(1)

        text_task = random_task['problem'].iloc[0]
        answer_task = random_task['answer'].iloc[0]

        return {
            "problem": text_task,
            "ground_truth": answer_task
        }

        

In [21]:
from typing_extensions import TypedDict
from typing import Annotated, Any
from langgraph.graph import StateGraph, END  # Убрал START
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

# from agents.generator import GeneratorAgent
# from agents.solver import SolverAgent
# from agents.reviewer import ReviewerAgent

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str
    problem: str
    ground_truth: str
    user_solution: str
    solver_answer: str
    solver_full: str
    review_verdict: str
    is_correct: bool

class MathWorkflow:
    def __init__(self):
        self.generator = GeneratorAgent()
        self.solver = SolverAgent()
        self.reviewer = ReviewerAgent()
        
        # Строим граф
        builder = StateGraph(AgentState)
        
        builder.add_node("generate", self._generate_node)
        builder.add_node("solve", self._solve_node)
        builder.add_node("review", self._review_node)
        
        # Используем строку "__start__" вместо START
        #builder.set_entry_point("generate")
        #builder.add_edge("generate", "solve")
        #builder.add_edge("solve", "review")
        #builder.add_edge("review", END)

        builder.set_entry_point("solve")
        builder.add_edge("solve", "review")
        builder.add_edge("review", END)
        
        self.graph = builder.compile()
    
    def _generate_node(self, state: AgentState) -> dict:
        result = self.generator.generate(state["topic"])
        return {
            "problem": result["problem"],
            "ground_truth": result["ground_truth"],
            "messages": []
        }
    
    def _solve_node(self, state: AgentState) -> dict:
        result = self.solver.solve(state["problem"])
        return {
            "solver_answer": result["answer"],
            "solver_full": result["full_response"],
            "messages": []
        }
    
    def _review_node(self, state: AgentState) -> dict:
        result = self.reviewer.review(
            solver_answer=state["user_solution"],
            ground_truth=state["ground_truth"]
        )
        return result
        #     {
        #     "review_verdict": result["verdict"],
        #     "is_correct": result["is_correct"],
        #     "messages": []
        # }
    
    def generate_only(self, topic: str) -> dict:
        """Только генерация задачи (для первого эндпоинта)"""
        return self.generator.generate(topic)
    
    def generate_only_static(self, topic: str) -> dict:
        """Только генерация задачи (для первого эндпоинта)"""
        return self.generator.generate_static_task(topic)
    
    def full_pipeline(self, topic: str, problem: str, user_solution: str, ground_truth: str) -> dict:
        """Полный pipeline: решение + проверка"""
        initial_state = {
            "messages": [],
            "topic": topic,
            "problem": problem,
            "ground_truth": ground_truth,
            "user_solution": user_solution,
            "solver_answer": "",
            "solver_full": "",
            "review_verdict": "",
            "is_correct": False
        }
        
        result = self.graph.invoke(initial_state)
        return result

In [22]:
wf = MathWorkflow()

In [6]:
result = wf.full_pipeline(
            topic="проверка",
            problem="Решить систему уравнений: x ³ – y ³ = 26, x ² y – xy ² = 6.",
            user_solution="Бла бла бла",
            ground_truth="(3, 1), (–1, –3)" or ""
        )

In [7]:
result

{'messages': [],
 'topic': 'проверка',
 'problem': 'Решить систему уравнений: x ³ – y ³ = 26, x ² y – xy ² = 6.',
 'ground_truth': '(3, 1), (–1, –3)',
 'user_solution': 'Бла бла бла',
 'solver_answer': '[числовой ответ]<|im_end|>',
 'solver_full': '<|startoftext|><|im_start|>system\nТы генератор математических задач. Придумай ОДНУ задачу по указанной теме.\n\nТребования:\n1. Задача должна быть реалистичной и понятной\n2. Ответ — число или простая формула\n3. Сложность: уровень ЕГЭ или школьной олимпиады\n\nФормат ответа (строго соблюдай):\nЗАДАЧА: [текст задачи]\nОТВЕТ: [числовой ответ]<|im_end|>\n<|im_start|>user\nРеши задачу: Решить систему уравнений: x ³ – y ³ = 26, x ² y – xy ² = 6.<|im_end|>\n<|im_start|>assistant\nЗАДАCHA: найти значения x и y, удовлетворяющие уравнениям  \nx³ − y³ = 26,  \nx² y − x y² = 6;  \nОТВЕТ: x + y  \n\nРассмотрим выражение (x - y)(x² + xy + y²) = 26 и уравнение x² y − x y² = xy(x−y) = 6.   \nПoserтивно делить второе уравнение на первое:  \n[xy(x−y)] / [(

In [26]:
ttt = AgentState(
    user_solution="бла бла бла",
    ground_truth="(3, 1), (–1, –3)"
)

In [27]:
res = wf._review_node(ttt)

In [31]:
res

{'full_response': '<|startoftext|><|im_start|>system\nТы генератор математических задач. Придумай ОДНУ задачу по указанной теме.\n\nТребования:\n1. Задача должна быть реалистичной и понятной\n2. Ответ — число или простая формула\n3. Сложность: уровень ЕГЭ или школьной олимпиады\n\nФормат ответа (строго соблюдай):\nЗАДАЧА: [текст задачи]\nОТВЕТ: [числовой ответ]<|im_end|>\n<|im_start|>user\nОтвет ученика: бла бла бла\nПравильный ответ: (3, 1), (–1, –3)<|im_end|>\n<|im_start|>assistant\nЗАДАЧА: Функция \\( f(x) = ax^2 + bx + c \\) имеет точки зрения \\((3, 1)\\) и \\((-1, -3)\\). Найдите пару коэффициентов \\((a,b,c)\\).\n\nОТВЕТ: (3, 1), (-1, –3) → ожидается уравнение функции, но из задания вероятно ошибко указано с точками. Просто для формата задачи подразумевается рассмотреть параметризацию решений на основании данных точек. Указанные конец предполагают несущественную сложность.\n\nБолее точным вариантом задачи подходит отдельное выражение. Например:\n\nЗАДАЧА: Вы установили систему п

In [33]:
print(res.content)

AttributeError: 'dict' object has no attribute 'content'

In [32]:
"CORRECT" in res['full_response']

False

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str
    problem: str
    ground_truth: str
    user_solution: str
    solver_answer: str
    solver_full: str
    review_verdict: str
    is_correct: bool

In [45]:
from langchain_core.messages import SystemMessage, HumanMessage
from config import get_llm

SYSTEM_PROMPT = """Ты проверяющий. Сравни ответ ученика с правильным ответом.

Ответь одним словом:
ПРАВИЛЬНО — если ответы совпадают (допускаются небольшие отличия в формате)
НЕПРАВИЛЬНО — если ответы разные

Не объясняй, просто скажи ПРАВИЛЬНО или НЕПРАВИЛЬНО."""

class ReviewerAgent:
    def __init__(self):
        self.llm = get_llm()
    
    def review(self, solver_answer: str, ground_truth: str = None) -> dict:
        gt_text = ground_truth if ground_truth else "не предоставлен"
        
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"Ответ ученика: {solver_answer}\nПравильный ответ: {gt_text}")
        ]
        
        response = self.llm.invoke(messages)
        return response
        # text = response.content.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip().upper()
        
        # is_correct = "ПРАВИЛЬНО" in text or "CORRECT" in text
        
        # return {
        #     "full_response": response.content,
        #     "verdict": "ПРАВИЛЬНО" if is_correct else "НЕПРАВИЛЬНО",
        #     "is_correct": is_correct
        # }

In [53]:
ra = ReviewerAgent()

In [54]:
res = ra.review(
    solver_answer="бла бла бла",
    ground_truth="(3, 1), (–1, –3)"
)

In [55]:
print(res.content)

<|startoftext|><|im_start|>system
Ты проверяющий. Сравни ответ ученика с правильным ответом.

Ответь одним словом:
ПРАВИЛЬНО — если ответы совпадают (допускаются небольшие отличия в формате)
НЕПРАВИЛЬНО — если ответы разные

Не объясняй, просто скажи ПРАВИЛЬНО или НЕПРАВИЛЬНО.<|im_end|>
<|im_start|>user
Ответ ученика: бла бла бла
Правильный ответ: (3, 1), (–1, –3)<|im_end|>
<|im_start|>assistant
НЕПРАВИЛЬНО


In [57]:
text = res.content.replace("<|system|>", "").replace("<|user|>", "").replace("<|assistant|>", "").strip().upper()

is_correct = "ПРАВИЛЬНО" in text or "CORRECT" in text

In [58]:
is_correct

True